# ✈️ FlightOps AI — Multi-Agent Flight Disruption Management

A **3-agent system** (under one orchestrator) that detects flight disruptions, rebooks passengers, and rules on regulatory compensation.

**Skills shown:** multi-agent orchestration · tool/function calling · RAG · structured outputs (Pydantic) · eval harness · cost/token observability.

Runs **offline (mock mode)** out of the box. Add an `ANTHROPIC_API_KEY` (Colab Secrets 🔑) to run live.

> Self-contained notebook — every cell below is runnable top to bottom on Colab.

## 1. Setup

In [ ]:
!pip install -q anthropic pydantic

import os
# Optional: pull API key from Colab Secrets (key icon in left sidebar). Mock mode if absent.
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('✅ API key loaded — running LIVE')
except Exception:
    print('ℹ️ No API key — running in MOCK mode (fully functional, deterministic)')

## 2. Schemas — structured outputs (Pydantic)
Every agent returns *validated* JSON, not free text.

In [ ]:
from __future__ import annotations
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field

class Severity(str, Enum):
    MINOR = 'minor'; MODERATE = 'moderate'; SEVERE = 'severe'

class DisruptionType(str, Enum):
    DELAY = 'delay'; CANCELLATION = 'cancellation'; DIVERSION = 'diversion'

class FlightDisruption(BaseModel):
    flight_no: str; origin: str; destination: str; scheduled_dep: str
    disruption_type: DisruptionType; delay_minutes: int = 0
    passengers: int = Field(ge=0); reason: Optional[str] = None

class DisruptionAssessment(BaseModel):
    severity: Severity; root_cause: str; weather_factor: bool
    estimated_resolution_minutes: int; affected_passengers: int; rationale: str

class AlternativeFlight(BaseModel):
    flight_no: str; carrier: str; dep_time: str; arr_time: str
    seats_available: int; extra_cost_usd: float

class RebookingPlan(BaseModel):
    alternatives: list[AlternativeFlight]; recommended_flight_no: str
    rebooking_notes: str; estimated_cost_usd: float

class ComplianceRuling(BaseModel):
    regulation_id: str; compensation_owed: bool; compensation_per_pax_usd: float
    care_obligations: list[str]; citation: str; reasoning: str

class FinalResolution(BaseModel):
    disruption: FlightDisruption; assessment: DisruptionAssessment
    rebooking: RebookingPlan; compliance: ComplianceRuling
    total_estimated_cost_usd: float; executive_summary: str

print('Schemas ready.')

## 3. Tools — function calling backends
Mock data so it runs offline; swap for NOAA / GDS / PSS APIs in production.

In [ ]:
import random

def get_weather(airport_code: str) -> dict:
    conditions = {
        'JFK': {'condition': 'snow', 'visibility_km': 1.2, 'wind_kts': 35, 'severe': True},
        'ORD': {'condition': 'thunderstorm', 'visibility_km': 3.0, 'wind_kts': 28, 'severe': True},
        'LAX': {'condition': 'clear', 'visibility_km': 16.0, 'wind_kts': 8, 'severe': False},
        'ATL': {'condition': 'rain', 'visibility_km': 6.0, 'wind_kts': 14, 'severe': False},
    }
    return conditions.get(airport_code.upper(),
                          {'condition': 'clear', 'visibility_km': 16.0, 'wind_kts': 6, 'severe': False})

def search_alternative_flights(origin: str, destination: str, after_time: str) -> list[dict]:
    carriers = ['DL', 'AA', 'UA', 'B6']; out = []; base_hour = 14
    for i in range(3):
        out.append({'flight_no': f'{random.choice(carriers)}{random.randint(100,999)}',
                    'carrier': random.choice(carriers),
                    'dep_time': f'{base_hour+i*2:02d}:30', 'arr_time': f'{base_hour+i*2+4:02d}:45',
                    'seats_available': random.randint(5, 60),
                    'extra_cost_usd': round(random.uniform(0, 220), 2)})
    return out

def rebook_passengers(flight_no: str, passenger_count: int) -> dict:
    return {'status': 'confirmed', 'flight_no': flight_no, 'rebooked': passenger_count,
            'confirmation': f'RBK{random.randint(10000,99999)}'}

TOOL_SCHEMAS = [
    {'name': 'get_weather', 'description': 'Get current weather at an airport by IATA code.',
     'input_schema': {'type': 'object', 'properties': {'airport_code': {'type': 'string'}}, 'required': ['airport_code']}},
    {'name': 'search_alternative_flights', 'description': 'Search alternative flights between two airports after a time.',
     'input_schema': {'type': 'object', 'properties': {'origin': {'type': 'string'}, 'destination': {'type': 'string'}, 'after_time': {'type': 'string'}}, 'required': ['origin', 'destination', 'after_time']}},
    {'name': 'rebook_passengers', 'description': 'Rebook passengers onto a target flight.',
     'input_schema': {'type': 'object', 'properties': {'flight_no': {'type': 'string'}, 'passenger_count': {'type': 'integer'}}, 'required': ['flight_no', 'passenger_count']}},
]
TOOL_REGISTRY = {'get_weather': get_weather, 'search_alternative_flights': search_alternative_flights, 'rebook_passengers': rebook_passengers}
def dispatch(name, args): return TOOL_REGISTRY[name](**args)
print('Tools ready.')

## 4. RAG knowledge base — aviation regulations
Token-overlap retrieval (zero-dependency). Swap for embeddings (e.g. voyage-3) in prod.

In [ ]:
import re
from dataclasses import dataclass

@dataclass
class Doc:
    id: str; title: str; text: str

KNOWLEDGE_BASE = [
    Doc('EU261-Art7', 'EU261 Article 7 — Right to Compensation',
        'Under EU Regulation 261/2004, passengers are entitled to compensation of 250 EUR for flights up to 1500 km, 400 EUR for intra-EU flights over 1500 km and other flights 1500-3500 km, and 600 EUR for all other flights, when a cancellation or long delay of 3 hours or more occurs and is within the carrier control. Weather and other extraordinary circumstances exempt the carrier from compensation but not from the duty of care.'),
    Doc('EU261-Art9', 'EU261 Article 9 — Right to Care',
        'Passengers facing delay are entitled to meals and refreshments, hotel accommodation when an overnight stay is required, transport between airport and accommodation, and two free communications. These care obligations apply even under extraordinary circumstances such as severe weather.'),
    Doc('USDOT-Refund-2024', 'US DOT Automatic Refund Rule',
        'US Department of Transportation rules require airlines to provide automatic cash refunds when a flight is cancelled or significantly changed and the passenger does not accept rebooking. A significant delay is generally 3 hours domestic or 6 hours international. Compensation beyond refunds is not federally mandated for weather-related disruptions.'),
    Doc('IATA-Reaccommodation', 'IATA Re-accommodation Guidance',
        'Carriers should re-accommodate passengers on the next available flight, including interline partners, prioritizing passengers with onward connections and those with reduced mobility. Cost minimization should not override duty of care obligations.'),
]
def _tokens(s): return set(re.findall(r'[a-z0-9]+', s.lower()))
def retrieve(query, k=2):
    q = _tokens(query)
    scored = sorted(((len(q & _tokens(d.text + ' ' + d.title)), d) for d in KNOWLEDGE_BASE), key=lambda x: x[0], reverse=True)
    return [d for s, d in scored[:k] if s > 0] or [KNOWLEDGE_BASE[0]]
print('RAG store ready.')

## 5. LLM client — live API or mock fallback
Includes a token/cost tracker for observability.

In [ ]:
import json
MODEL = 'claude-sonnet-4-6'
_LIVE = bool(os.environ.get('ANTHROPIC_API_KEY'))
try:
    from anthropic import Anthropic
    _client = Anthropic() if _LIVE else None
except Exception:
    _client = None; _LIVE = False

class TokenTracker:
    def __init__(self): self.input_tokens = 0; self.output_tokens = 0
    def add(self, u):
        if u: self.input_tokens += getattr(u, 'input_tokens', 0); self.output_tokens += getattr(u, 'output_tokens', 0)
    @property
    def est_cost_usd(self): return round(self.input_tokens/1e6*3 + self.output_tokens/1e6*15, 4)
TRACKER = TokenTracker()

def call(system, messages, tools=None, mock_fn=None):
    if not _LIVE or _client is None:
        return mock_fn(system, messages) if mock_fn else {'text': '{}', 'tool_calls': [], 'stop_reason': 'end_turn'}
    kwargs = dict(model=MODEL, max_tokens=1024, system=system, messages=messages)
    if tools: kwargs['tools'] = tools
    resp = _client.messages.create(**kwargs); TRACKER.add(resp.usage)
    text, tool_calls = '', []
    for b in resp.content:
        if b.type == 'text': text += b.text
        elif b.type == 'tool_use': tool_calls.append({'id': b.id, 'name': b.name, 'input': b.input})
    return {'text': text, 'tool_calls': tool_calls, 'stop_reason': resp.stop_reason}

def extract_json(text):
    text = text.strip().replace('```json', '').replace('```', '').strip()
    s, e = text.find('{'), text.rfind('}')
    if s == -1 or e == -1: return {}
    try: return json.loads(text[s:e+1])
    except json.JSONDecodeError: return {}
print(f'LLM client ready — mode: {"LIVE" if _LIVE else "MOCK"}')

## 6. The 3 Agents

In [ ]:
# --- Agent 1: Disruption Analyst (tool calling) ---
ANALYST_SYS = ('You are a Disruption Analyst. Call get_weather for the origin, then assess severity '
    'and root cause. Respond ONLY with JSON: {"severity":"minor|moderate|severe","root_cause":str,'
    '"weather_factor":bool,"estimated_resolution_minutes":int,"affected_passengers":int,"rationale":str}')

def analyst_run(d: FlightDisruption) -> DisruptionAssessment:
    msgs = [{'role': 'user', 'content': f'Disruption: {d.model_dump_json()}. Assess it.'}]
    weather = None; resp = {'text': ''}
    def mock(s, m):
        if isinstance(m[-1]['content'], list): return {'text': '', 'tool_calls': [], 'stop_reason': 'end_turn'}
        return {'text': '', 'stop_reason': 'tool_use', 'tool_calls': [{'id': 't1', 'name': 'get_weather', 'input': {'airport_code': d.origin}}]}
    for _ in range(3):
        resp = call(ANALYST_SYS, msgs, tools=TOOL_SCHEMAS, mock_fn=mock)
        if resp.get('tool_calls'):
            tc = resp['tool_calls'][0]
            if tc['name'] == 'get_weather': tc['input']['airport_code'] = d.origin
            res = dispatch(tc['name'], tc['input'])
            if tc['name'] == 'get_weather': weather = res
            msgs.append({'role': 'assistant', 'content': [{'type': 'tool_use', 'id': tc['id'], 'name': tc['name'], 'input': tc['input']}]})
            msgs.append({'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': tc['id'], 'content': str(res)}]})
            continue
        break
    data = extract_json(resp.get('text', ''))
    if not data:
        weather = weather or get_weather(d.origin); wx = weather.get('severe', False); mins = d.delay_minutes
        sev = Severity.SEVERE if mins >= 180 or d.disruption_type.value == 'cancellation' else Severity.MODERATE if mins >= 60 else Severity.MINOR
        data = {'severity': sev.value, 'root_cause': f"{weather['condition']} at {d.origin}" if wx else (d.reason or 'operational'),
                'weather_factor': wx, 'estimated_resolution_minutes': max(mins, 90 if wx else 45),
                'affected_passengers': d.passengers, 'rationale': f'{d.delay_minutes}min disruption; weather severe={wx}.'}
    return DisruptionAssessment(**data)

# --- Agent 2: Rebooking (cost optimization) ---
def rebooking_run(d: FlightDisruption, a: DisruptionAssessment) -> RebookingPlan:
    alts = [AlternativeFlight(**x) for x in search_alternative_flights(d.origin, d.destination, d.scheduled_dep)]
    viable = [x for x in alts if x.seats_available >= d.passengers]
    if viable:
        best = min(viable, key=lambda x: x.extra_cost_usd)
        notes = 'Single flight covers all passengers at lowest incremental cost.'; cost = best.extra_cost_usd * d.passengers
    else:
        best = max(alts, key=lambda x: x.seats_available)
        notes = 'No single flight has enough seats; split load, prioritizing connections and reduced-mobility pax per IATA.'
        cost = sum(x.extra_cost_usd * x.seats_available for x in alts[:2])
    return RebookingPlan(alternatives=alts, recommended_flight_no=best.flight_no, rebooking_notes=notes, estimated_cost_usd=round(cost, 2))

# --- Agent 3: Compliance (RAG) ---
COMPLIANCE_SYS = ('You are an airline Compliance Officer. Using ONLY the provided regulation excerpts, '
    'determine compensation + care obligations. Weather exempts compensation but NOT duty of care. '
    'Respond ONLY with JSON: {"regulation_id":str,"compensation_owed":bool,"compensation_per_pax_usd":number,'
    '"care_obligations":[str],"citation":str,"reasoning":str}')

def compliance_run(d: FlightDisruption, a: DisruptionAssessment) -> ComplianceRuling:
    docs = retrieve(f'{d.disruption_type.value} delay {d.delay_minutes} weather {a.weather_factor} compensation care', k=2)
    ctx = '\n\n'.join(f'[{x.id}] {x.title}\n{x.text}' for x in docs)
    msgs = [{'role': 'user', 'content': f'Regulations:\n{ctx}\n\nDisruption: {d.model_dump_json()}\nWeather-caused: {a.weather_factor}, delay: {d.delay_minutes}min. Rule on it.'}]
    resp = call(COMPLIANCE_SYS, msgs, mock_fn=lambda s, m: {'text': '', 'stop_reason': 'end_turn'})
    data = extract_json(resp.get('text', ''))
    if not data:
        long_delay = d.delay_minutes >= 180 or d.disruption_type.value == 'cancellation'
        owed = long_delay and not a.weather_factor; comp = 600.0 if owed else 0.0
        care = ['meals/refreshments', 'hotel if overnight', 'ground transport', 'two free communications'] if d.delay_minutes >= 120 else []
        data = {'regulation_id': docs[0].id, 'compensation_owed': owed, 'compensation_per_pax_usd': comp,
                'care_obligations': care, 'citation': f'{docs[0].id}: {docs[0].title}',
                'reasoning': f"{'Long delay/cancellation' if long_delay else 'Short delay'}; weather_factor={a.weather_factor}. duty of care {'applies' if care else 'not triggered'}."}
    return ComplianceRuling(**data)
print('Agents ready.')

## 7. Orchestrator — coordinates the 3 agents
Sequences agents, passes state, aggregates one validated result, traces timing.

In [ ]:
import time
def resolve(d: FlightDisruption) -> FinalResolution:
    print(f'\n🛫 Orchestrating {d.flight_no} ({d.origin}→{d.destination})')
    t = time.time(); a = analyst_run(d); print(f'  [trace] disruption_analyst  {(time.time()-t)*1000:6.1f} ms')
    t = time.time(); r = rebooking_run(d, a); print(f'  [trace] rebooking_agent     {(time.time()-t)*1000:6.1f} ms')
    t = time.time(); c = compliance_run(d, a); print(f'  [trace] compliance_agent    {(time.time()-t)*1000:6.1f} ms')
    total = round(r.estimated_cost_usd + c.compensation_per_pax_usd * d.passengers, 2)
    summary = (f'{d.disruption_type.value.title()} on {d.flight_no} rated {a.severity.value.upper()} '
        f'(cause: {a.root_cause}). Rebook {d.passengers} pax onto {r.recommended_flight_no}. '
        f'Compensation owed: {c.compensation_owed} (${c.compensation_per_pax_usd:.0f}/pax). Total est. cost: ${total:,.2f}.')
    return FinalResolution(disruption=d, assessment=a, rebooking=r, compliance=c, total_estimated_cost_usd=total, executive_summary=summary)
print('Orchestrator ready.')

## 8. Run the demo 🎬

In [ ]:
scenario = FlightDisruption(flight_no='DL1423', origin='JFK', destination='LAX', scheduled_dep='14:00',
                            disruption_type=DisruptionType.CANCELLATION, delay_minutes=240, passengers=180, reason='winter storm')
result = resolve(scenario)
print('\n📋 RESOLUTION\n')
print(json.dumps(result.model_dump(), indent=2, default=str))
print('\n📝', result.executive_summary)

## 9. Eval harness — automated correctness checks ✅
Shows you treat agents like software, not magic. This is a big interview differentiator.

In [ ]:
CASES = [
    ('weather_cancellation_no_compensation',
     FlightDisruption(flight_no='DL1423', origin='JFK', destination='LAX', scheduled_dep='14:00', disruption_type=DisruptionType.CANCELLATION, delay_minutes=240, passengers=180, reason='winter storm'),
     lambda r: r.assessment.severity == Severity.SEVERE and r.assessment.weather_factor and not r.compliance.compensation_owed and len(r.compliance.care_obligations) > 0),
    ('non_weather_long_delay_owes_compensation',
     FlightDisruption(flight_no='UA200', origin='LAX', destination='ATL', scheduled_dep='08:00', disruption_type=DisruptionType.DELAY, delay_minutes=200, passengers=120, reason='crew scheduling'),
     lambda r: r.compliance.compensation_owed and r.compliance.compensation_per_pax_usd > 0),
    ('minor_delay_no_obligations',
     FlightDisruption(flight_no='AA887', origin='LAX', destination='ATL', scheduled_dep='09:30', disruption_type=DisruptionType.DELAY, delay_minutes=40, passengers=140, reason='late inbound aircraft'),
     lambda r: r.assessment.severity == Severity.MINOR and not r.compliance.compensation_owed),
    ('rebooking_recommends_valid_flight',
     FlightDisruption(flight_no='B655', origin='JFK', destination='LAX', scheduled_dep='12:00', disruption_type=DisruptionType.DELAY, delay_minutes=190, passengers=30, reason='mechanical'),
     lambda r: r.rebooking.recommended_flight_no in [a.flight_no for a in r.rebooking.alternatives] and r.total_estimated_cost_usd >= 0),
]
passed = 0
for name, inp, check in CASES:
    try: ok = bool(check(resolve(inp)))
    except Exception as e: ok = False; print('  ERROR', name, e)
    passed += ok; print(f"  {'✅ PASS' if ok else '❌ FAIL'}  {name}")
print(f'\n{passed}/{len(CASES)} cases passed ({100*passed//len(CASES)}%)')
if _LIVE: print(f'\n💰 Run cost: ${TRACKER.est_cost_usd}')